In [ ]:
pip install -r requirements.txt

In [ ]:
!chcp 65001

In [ ]:
!python data_load.py --root data/ --out data/df_raw.parquet

In [ ]:
import pandas as pd
df = pd.read_parquet("data/df_raw.parquet")
print(df.columns.tolist())

In [ ]:
!python feature_engineering.py \
       --raw data/df_raw.parquet \
       --out data/df_features.parquet \
       --pipe models/preprocess.joblib

In [ ]:
!python cond.py --raw data/df_raw.parquet

In [ ]:

%matplotlib inline
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import chi2_contingency
from pathlib import Path

plt.style.use("seaborn-v0_8-darkgrid")

df = pd.read_parquet(Path("data") / "df_features.parquet")
print(f"Rows: {len(df):,}  -  Cols: {df.shape[1]}")

display(df.head())

with pd.option_context("display.max_rows", None):
    display(df.describe(include="all").T)
print(df.info(memory_usage="deep"))

ax = df["target"].value_counts().rename({0: "Неуспешный", 1: "Успешный"}).plot(kind="bar")
ax.set(ylabel="Количество", title="Распределение успешности")
plt.tight_layout(); plt.show()

skip_cols = {"student_uuid", "target"}
num_cols  = [c for c, dt in df.dtypes.items() if pd.api.types.is_numeric_dtype(dt) and c not in skip_cols]

corr = df[num_cols + ["target"]].corr()
plt.figure(figsize=(1.2*len(num_cols), 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Кореляция численных критериев и успешности")
plt.show()

cat_cols = df.select_dtypes(["object", "category"]).columns.tolist()
for c in cat_cols:
    if df[c].nunique() > 1:
        ct = pd.crosstab(df[c], df["target"])
        chi2, p, *_ = chi2_contingency(ct)
        print(f"{c:25s}  p-value: {p:.3e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
if "gpa_1_4" in df.columns:
    sns.boxplot(data=df, x="target", y="gpa_1_4", ax=axes[0])
    axes[0].set_title("Средняя оценка за 1-4 семестр и успешные студенты")
else:
    axes[0].set_visible(False)

fail_col = next((c for c in ["n_3", "n_4", "fail_cnt_1_4"] if c in df.columns), None)
if fail_col:
    sns.boxplot(data=df, x="target", y=fail_col, ax=axes[1])
    axes[1].set_title(f"{fail_col} и успешность")
else:
    axes[1].set_visible(False)
plt.tight_layout(); plt.show()

if {"group_size", "target"}.issubset(df.columns):
    sample = df.sample(min(100_000, len(df)), random_state=42)
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x="group_size", y="target", data=sample, alpha=0.1)
    sns.regplot(x="group_size", y="target", data=sample,
                logistic=True, scatter=False, color="red")
    plt.title("Шанс успешности и размер группы")
    plt.show()

na_share = df.isna().mean().sort_values(ascending=False)
display(na_share[na_share > 0].head(10))

try:
    from ydata_profiling import ProfileReport
    sample_df = df.sample(min(len(df), 200_000), random_state=42)
    profile = ProfileReport(sample_df, title="EDA report", explorative=True)
    profile.to_file("eda_report.html")
    print("EDA report saved → eda_report.html")
except ImportError:
    warnings.warn("ydata_profiling не установлен — пропускаю шаг профилирования")

In [ ]:
df = pd.read_parquet("data/df_raw.parquet")

print(df["final_state"].value_counts())
print(pd.crosstab(df["final_state"], df["target"], normalize="index"))

In [ ]:
import xgboost; print(xgboost.__version__)

In [ ]:
!python train_baseline.py \
        --data   data/df_features.parquet \
        --pipe   models/preprocess.joblib \
        --model  models/best_model.joblib \
        --report reports/metrics_baseline.csv

In [ ]:
pip install shap

In [ ]:
!python tune_and_interpret.py --n-trials 1